# Colab 9 — Siamese on Synthetic AA-alphabet Data (length 8)

**Goal:** validate the colab8 Siamese architecture on a 20-letter alphabet *before* committing to real protein data.

We use the **standard 20 amino acid letters** (`ACDEFGHIKLMNPQRSTVWY`) but generate sequences uniformly at random — this isolates the architecture/scaling question from the biology. If this works, we move to Swiss-Prot in colab10.

**What changes from colab8:**
- `vocab_size`: 2 → 20
- `embed_dim`: 16 → 32 (richer alphabet needs more capacity per token)
- Tokenization: char → int via lookup table (instead of `int(c)`)
- **Pair sampling is now hybrid** — can't enumerate 20^8 ≈ 26B sequences. We sample a corpus, then build pairs from a mix of (a) random pairs and (b) seed+k-edit perturbations. This is essential because uniform random pairs over 20 letters almost always have d≈8 — without perturbation pairs, the model gets no training signal at small distances.

**Architecture (unchanged otherwise):**
```
  seq_A ──► Embed(20,32) → Conv1D(32→32) → Conv1D(32→64) → AvgPool → Linear(64→128) → L2Norm ──► emb_A
```

**Success criteria:** Spearman ρ > 0.85, Recall@5 > 0.7 on a held-out query set.

## 1. Setup & Imports

In [ ]:
import os
os.chdir('/content')
!rm -rf thesis-edit-distance-nn
!git clone https://github.com/katzemelli/thesis-edit-distance-nn.git
os.chdir('/content/thesis-edit-distance-nn')

In [ ]:
!pip install torch torchvision --quiet

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
from scipy.stats import pearsonr, spearmanr
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 2. Alphabet & Levenshtein

Standard 20 amino acid letters. We map each letter to an integer index for the embedding layer.

In [ ]:
AA_ALPHABET = 'ACDEFGHIKLMNPQRSTVWY'
VOCAB_SIZE = len(AA_ALPHABET)
SEQ_LEN = 8

CHAR_TO_IDX = {c: i for i, c in enumerate(AA_ALPHABET)}
IDX_TO_CHAR = {i: c for c, i in CHAR_TO_IDX.items()}

print(f'Alphabet: {AA_ALPHABET}')
print(f'Vocab size: {VOCAB_SIZE}')
print(f'Sequence length: {SEQ_LEN}')
print(f'Total possible sequences: {VOCAB_SIZE ** SEQ_LEN:,} (cannot enumerate)')

def encode(seq):
    """Convert AA string to tensor of indices."""
    return torch.tensor([CHAR_TO_IDX[c] for c in seq], dtype=torch.long)

def levenshtein(a, b):
    """Standard Wagner-Fischer DP — alphabet-agnostic."""
    m, n = len(a), len(b)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            cost = 0 if a[i-1] == b[j-1] else 1
            dp[i][j] = min(dp[i-1][j] + 1, dp[i][j-1] + 1, dp[i-1][j-1] + cost)
    return dp[m][n]

## 3. Synthetic Corpus & Hybrid Pair Sampling

Two-part data generation:
1. **Corpus:** N random length-8 sequences over the AA alphabet.
2. **Pairs:**
   - 50% **random pairs** (corpus[i], corpus[j]) — gives the high-distance signal (mostly d≈8).
   - 50% **perturbation pairs** (seed, seed+k_edits) for k∈{0,1,2,3,4} — gives the low-distance signal where retrieval lives.

Each edit is one of: substitution, insertion, deletion. We re-pad/truncate to length 8 so the network sees fixed-length input. The *true* Levenshtein is recomputed by DP — we don't assume k edits = distance k.

In [ ]:
rng = np.random.default_rng(SEED)

def random_seq(length=SEQ_LEN):
    return ''.join(rng.choice(list(AA_ALPHABET), size=length))

def perturb(seq, k):
    """Apply k random edits (sub/ins/del), then pad/truncate back to length SEQ_LEN."""
    s = list(seq)
    for _ in range(k):
        op = rng.choice(['sub', 'ins', 'del'])
        if len(s) == 0:
            op = 'ins'
        if op == 'sub':
            i = rng.integers(0, len(s))
            choices = [c for c in AA_ALPHABET if c != s[i]]
            s[i] = rng.choice(choices)
        elif op == 'ins':
            i = rng.integers(0, len(s) + 1)
            s.insert(i, rng.choice(list(AA_ALPHABET)))
        elif op == 'del':
            i = rng.integers(0, len(s))
            del s[i]
    # Re-pad / truncate to fixed length
    if len(s) < SEQ_LEN:
        s = s + [rng.choice(list(AA_ALPHABET)) for _ in range(SEQ_LEN - len(s))]
    elif len(s) > SEQ_LEN:
        s = s[:SEQ_LEN]
    return ''.join(s)

# 1. Build corpus
CORPUS_SIZE = 5000
corpus = [random_seq() for _ in range(CORPUS_SIZE)]
corpus = list(set(corpus))  # dedupe (collisions extremely rare)
print(f'Corpus size: {len(corpus)}')
print(f'Examples: {corpus[:5]}')

# 2. Build training pairs
N_PAIRS = 40000
n_random = N_PAIRS // 2
n_perturb = N_PAIRS - n_random

pairs = []

# (a) Random pairs from corpus
for _ in range(n_random):
    i, j = rng.integers(0, len(corpus), size=2)
    a, b = corpus[i], corpus[j]
    d = levenshtein(a, b)
    pairs.append((a, b, d / SEQ_LEN))

# (b) Perturbation pairs
for _ in range(n_perturb):
    seed = corpus[rng.integers(0, len(corpus))]
    k = int(rng.integers(0, 5))  # 0..4 edits
    perturbed = perturb(seed, k)
    d = levenshtein(seed, perturbed)
    pairs.append((seed, perturbed, d / SEQ_LEN))

print(f'Total pairs: {len(pairs)}  (random: {n_random}, perturbation: {n_perturb})')

# Plot the distance distribution — should NOT be all d≈8
norm_dists = [p[2] for p in pairs]
plt.figure(figsize=(8, 3))
plt.hist(norm_dists, bins=np.arange(0, 1.05, 1/SEQ_LEN), edgecolor='k', alpha=0.75)
plt.axvline(x=0.4, color='r', linestyle='--', label='oversample threshold')
plt.xlabel('Normalized Levenshtein distance')
plt.ylabel('Count')
plt.title('Training-pair distance distribution')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

dist_counts = Counter(round(d * SEQ_LEN) for d in norm_dists)
print('Pairs per (un-normalized) integer distance:')
for d in sorted(dist_counts.keys()):
    print(f'  d={d}: {dist_counts[d]} pairs')

## 4. PyTorch Dataset & DataLoader

Same close-pair oversampling as colab8 (3× at threshold 0.4). The hybrid sampling already over-represents close pairs, so the oversampling on top should push the loss firmly toward small-distance accuracy.

In [ ]:
class SequencePairDataset(Dataset):
    def __init__(self, pairs, oversample_threshold=0.4, oversample_factor=3):
        self.data = []
        for seq_a, seq_b, norm_dist in pairs:
            entry = (
                encode(seq_a),
                encode(seq_b),
                torch.tensor(norm_dist, dtype=torch.float32)
            )
            self.data.append(entry)
            if norm_dist < oversample_threshold:
                for _ in range(oversample_factor - 1):
                    self.data.append(entry)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

dataset = SequencePairDataset(pairs)
dataloader = DataLoader(dataset, batch_size=256, shuffle=True)

print(f'Dataset size (with oversampling): {len(dataset)}')
print(f'Original pairs: {len(pairs)}')
print(f'Oversampled close pairs added: {len(dataset) - len(pairs)}')

## 5. Siamese Encoder — `vocab_size=20`, `embed_dim=32`

Identical to colab8 except the two changes called out at the top. Total parameters should be a small bump from 16,128 (colab8) to ~22k.

In [ ]:
class SiameseEncoder(nn.Module):
    def __init__(self, vocab_size=VOCAB_SIZE, embed_dim=32, hidden1=32, hidden2=64, out_dim=128):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.conv1 = nn.Conv1d(embed_dim, hidden1, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(hidden1, hidden2, kernel_size=3, padding=1)
        self.fc = nn.Linear(hidden2, out_dim)

    def forward(self, x):
        x = self.embedding(x)
        x = x.permute(0, 2, 1)
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = x.mean(dim=2)
        x = self.fc(x)
        x = F.normalize(x, p=2, dim=1)
        return x


class SiameseNetwork(nn.Module):
    def __init__(self, vocab_size=VOCAB_SIZE, embed_dim=32, out_dim=128):
        super().__init__()
        self.encoder = SiameseEncoder(vocab_size, embed_dim, out_dim=out_dim)

    def forward(self, seq_a, seq_b):
        emb_a = self.encoder(seq_a)
        emb_b = self.encoder(seq_b)
        dist = torch.norm(emb_a - emb_b, p=2, dim=1)
        return dist

    def encode(self, seq):
        return self.encoder(seq)

model = SiameseNetwork().to(device)
print(model)
total_params = sum(p.numel() for p in model.parameters())
print(f'Total parameters: {total_params:,}')

## 6. Weighted MSE Loss

Same `weight = exp(-α · true_dist)` with α=3 as colab8. The distance histogram from cell 3 should tell us if α needs retuning — if most pairs sit at d=1.0, the weighting is even more important here.

In [ ]:
def weighted_mse_loss(pred_dist, true_dist, alpha=3.0):
    weight = torch.exp(-alpha * true_dist)
    return torch.mean(weight * (pred_dist - true_dist) ** 2)

d_vals = np.linspace(0, 1, 100)
w_vals = np.exp(-3.0 * d_vals)
plt.figure(figsize=(6, 3))
plt.plot(d_vals, w_vals)
plt.axvline(x=0.4, color='r', linestyle='--', label='threshold = 0.4')
plt.xlabel('Normalized Levenshtein distance')
plt.ylabel('Weight')
plt.title('Weighted MSE: weight = exp(-3 · d)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Training Loop

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
num_epochs = 100
losses = []

model.train()
for epoch in range(1, num_epochs + 1):
    epoch_loss = 0.0
    num_batches = 0
    for seq_a, seq_b, true_dist in dataloader:
        seq_a = seq_a.to(device)
        seq_b = seq_b.to(device)
        true_dist = true_dist.to(device)

        pred_dist = model(seq_a, seq_b)
        loss = weighted_mse_loss(pred_dist, true_dist)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        num_batches += 1

    avg_loss = epoch_loss / num_batches
    losses.append(avg_loss)
    if epoch % 5 == 0 or epoch == 1:
        print(f'Epoch {epoch:3d}/{num_epochs} — Loss: {avg_loss:.6f}')

plt.figure(figsize=(8, 4))
plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('Weighted MSE Loss')
plt.title('Training Loss — Synthetic AA, length 8')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print(f'Final loss: {losses[-1]:.6f}')

## 8. Held-out Evaluation Set

We can't enumerate all 20^8 sequences. Instead we evaluate on:
- An **eval corpus** of 1000 fresh random sequences (not used in training)
- A **query set** of 200 sequences drawn from the eval corpus

For each query, we compute the *true* k-NN over the eval corpus (by Levenshtein) and the *predicted* k-NN (by embedding L2 distance). Recall@k is the overlap.

In [ ]:
EVAL_CORPUS_SIZE = 1000
N_QUERIES = 200

eval_corpus = list(set(random_seq() for _ in range(EVAL_CORPUS_SIZE)))
print(f'Eval corpus size: {len(eval_corpus)}')

model.eval()
eval_tensors = torch.stack([encode(s) for s in eval_corpus]).to(device)
with torch.no_grad():
    eval_embeddings = model.encode(eval_tensors).cpu().numpy()
print(f'Eval embedding shape: {eval_embeddings.shape}')

# Pairwise embedding distance matrix
emb_tensor = torch.tensor(eval_embeddings, dtype=torch.float32)
emb_dists = torch.cdist(emb_tensor, emb_tensor, p=2).numpy()

## 9. Distance Correlation Plot

All unique pairs in the eval corpus: true normalized Levenshtein vs. embedding L2 distance. We expect a tight monotonic relationship; Spearman ρ is the metric to watch.

In [ ]:
n_eval = len(eval_corpus)
true_dists = []
pred_dists = []

# Subsample pairs (full pairwise = ~500k pairs is fine, but we cap for speed)
MAX_EVAL_PAIRS = 50000
all_ij = [(i, j) for i in range(n_eval) for j in range(i + 1, n_eval)]
if len(all_ij) > MAX_EVAL_PAIRS:
    sampled_idx = rng.choice(len(all_ij), size=MAX_EVAL_PAIRS, replace=False)
    all_ij = [all_ij[k] for k in sampled_idx]

for i, j in all_ij:
    true_dists.append(levenshtein(eval_corpus[i], eval_corpus[j]) / SEQ_LEN)
    pred_dists.append(emb_dists[i, j])

true_dists = np.array(true_dists)
pred_dists = np.array(pred_dists)

pearson_r, pearson_p = pearsonr(true_dists, pred_dists)
spearman_r, spearman_p = spearmanr(true_dists, pred_dists)
print(f'Pearson  r = {pearson_r:.4f} (p = {pearson_p:.2e})')
print(f'Spearman ρ = {spearman_r:.4f} (p = {spearman_p:.2e})')

plt.figure(figsize=(7, 6))
plt.scatter(true_dists, pred_dists, alpha=0.05, s=5)
coeffs = np.polyfit(true_dists, pred_dists, 1)
x_line = np.linspace(0, 1, 100)
plt.plot(x_line, np.polyval(coeffs, x_line), 'r-', linewidth=2,
         label=f'Linear fit (slope={coeffs[0]:.3f})')
plt.xlabel('True Normalized Levenshtein Distance')
plt.ylabel('Predicted Embedding L2 Distance')
plt.title(f'Distance Correlation — Synthetic AA, length 8\nPearson r={pearson_r:.3f}, Spearman ρ={spearman_r:.3f}')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 10. k-NN Evaluation

For each of the 200 queries, compute Recall@k against the eval corpus. The colab8 length-8 numbers were Recall@5=0.91, Recall@8=0.91 — but that was 256 binary strings. Here the corpus is 1000 sequences over 20 letters, so the retrieval problem is different. Targets:

- Recall@1 > 0.5
- Recall@5 > 0.7
- Recall@10 > 0.8

In [ ]:
# Pick query indices
query_idx = rng.choice(n_eval, size=min(N_QUERIES, n_eval), replace=False)

# True distance matrix (queries x corpus) — only compute the rows we need
print(f'Computing true Levenshtein for {len(query_idx)} queries × {n_eval} corpus...')
true_dist_rows = np.zeros((len(query_idx), n_eval))
for qi, q in enumerate(query_idx):
    for j in range(n_eval):
        true_dist_rows[qi, j] = levenshtein(eval_corpus[q], eval_corpus[j])

def get_knn_from_row(dist_row, self_idx, k):
    dists = dist_row.copy().astype(float)
    dists[self_idx] = np.inf
    return set(np.argsort(dists)[:k])

for k in [1, 5, 10]:
    recalls = []
    for qi, q in enumerate(query_idx):
        true_knn = get_knn_from_row(true_dist_rows[qi], q, k)
        pred_knn = get_knn_from_row(emb_dists[q], q, k)
        recalls.append(len(true_knn & pred_knn) / k)
    print(f'Mean Recall@{k}: {np.mean(recalls):.4f}')

## 11. Embedding Visualization (PCA / t-SNE)

Project the eval embeddings to 2D, color by Levenshtein distance to a single reference sequence. A clean radial gradient = the embedding has organized sequences by edit distance.

In [ ]:
ref_idx = 0
ref_seq = eval_corpus[ref_idx]
colors = [levenshtein(s, ref_seq) for s in eval_corpus]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

pca_2d = PCA(n_components=2).fit_transform(eval_embeddings)
sc1 = axes[0].scatter(pca_2d[:, 0], pca_2d[:, 1], c=colors, cmap='viridis', s=15, alpha=0.7)
axes[0].set_title('PCA projection')
axes[0].set_xlabel('PC1')
axes[0].set_ylabel('PC2')

tsne_2d = TSNE(n_components=2, perplexity=30, random_state=SEED).fit_transform(eval_embeddings)
sc2 = axes[1].scatter(tsne_2d[:, 0], tsne_2d[:, 1], c=colors, cmap='viridis', s=15, alpha=0.7)
axes[1].set_title('t-SNE projection')
axes[1].set_xlabel('t-SNE 1')
axes[1].set_ylabel('t-SNE 2')

fig.colorbar(sc2, ax=axes, label=f"Levenshtein distance to '{ref_seq}'")
plt.suptitle('Embedding Space — 1000 Synthetic AA Strings of Length 8', fontsize=13)
plt.tight_layout()
plt.show()

## 12. Summary & Next Step Decision

If Spearman ρ > 0.85 and Recall@5 > 0.7 → architecture is sound, move to colab10 with real Swiss-Prot data.

If metrics fall short:
- Spearman OK but Recall@k weak → embedding manifold is monotonic but not discriminative at small distances → bump α (loss weighting), more close-pair oversampling, or larger out_dim.
- Both weak → architecture under-capacity → bump embed_dim further (32→64), wider conv channels, or longer training.
- Loss plateau early → check distance histogram; if dominated by d≈1.0, increase perturbation share above 50%.